# BDC Doc RAG — agent with tool calling

Starts the API, then runs a [deepagents](https://github.com/langchain-ai/deepagents) agent
(same pattern as `langgraph_test/deep_agents.ipynb`): the configured LLM (`config.get_llm()`)
gets `search_docs` as a LangChain tool and decides when to call it.

Run top to bottom. The last cell shuts the API down.
Needs an ingested `.chroma_db` **and** a reachable completion provider (see `.env`).
The MCP server itself is covered separately by `tests/test_mcp.py`.

In [1]:
import socket, subprocess, sys, tempfile, time
from pathlib import Path

import httpx

# a live server keeps its old code and its old .env forever, so re-running this cell must
# replace the previous one rather than leak it and leave you querying a stale process
try:
    api.terminate()
    api.wait(timeout=10)
    print("replaced the previously started API")
except NameError:
    pass

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
with socket.socket() as s:  # let the OS pick a free port so an already-running API doesn't clash
    s.bind(("127.0.0.1", 0))
    PORT = s.getsockname()[1]
API_URL = f"http://127.0.0.1:{PORT}"

# a file, not a PIPE: nothing drains a pipe here, and uvicorn deadlocks once its buffer fills
LOG = Path(tempfile.gettempdir()) / f"bdc_api_{PORT}.log"
api = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "bdc_doc_mcp.api:app", "--port", str(PORT)],
    cwd=ROOT, stdout=LOG.open("w"), stderr=subprocess.STDOUT,
)

for _ in range(120):  # first request loads the embedding + chroma clients, so be patient
    if api.poll() is not None:
        raise RuntimeError(f"API died:\n{LOG.read_text()}")
    try:
        httpx.get(f"{API_URL}/health", timeout=2).raise_for_status()
        break
    except Exception:
        time.sleep(1)
else:
    api.terminate()
    raise RuntimeError(f"API never became healthy:\n{LOG.read_text()}")

print("API up at", API_URL, "| server log:", LOG)

API up at http://127.0.0.1:58496 | server log: C:\Users\steve\AppData\Local\Temp\bdc_api_58496.log


## Model + search tool

The tool wraps `POST /search`; the model comes from `.env` (azure / openai / vllm / ollama).

Why a hand-written `@tool` instead of loading the MCP tool via `langchain-mcp-adapters`:
the MCP stdio client needs a real-file `errlog` under ipykernel on Windows (`sys.stderr`
has no `fileno()`), which the adapters don't expose. The MCP server itself is covered by
`tests/test_mcp.py` — keep this tool's docstring in sync with `mcp_server.search_docs`.

In [ ]:
assert "API_URL" in globals(), "run the boot cell (first code cell) first — it starts the API"

sys.path.insert(0, str(ROOT))  # notebook runs from tests/, the package lives at the repo root

from langchain.tools import tool

from bdc_doc_mcp.config import get_llm

model = get_llm()


@tool
def search_docs(query: str, k: int = 5, doc_type: str | None = None,
                date_from: str | None = None, date_to: str | None = None) -> str:
    """Semantic search over the BDC (NHLBI BioData Catalyst) documentation database.

    Use this to look up BDC facts before answering. doc_type is a CSV string of types
    to search (e.g. "page,faq" or "video"); available types: page, faq, event, fellow,
    update, video, docs. When omitted, time-sensitive types (update, event) are
    excluded — name them explicitly to search news posts or events.

    date_from / date_to ("YYYY-MM-DD", inclusive) filter by date. Only event and
    update docs carry a date, so a date filter implicitly narrows to those types.
    Results are ranked by relevance, NOT date — for "recent"/"latest" questions,
    always set date_from to bound the range, then compare the dates returned.

    Args:
        query: natural-language search query.
        k: number of chunks to return.
        doc_type: CSV of document types to restrict the search to.
        date_from: earliest date to include ("YYYY-MM-DD").
        date_to: latest date to include ("YYYY-MM-DD").
    """
    resp = httpx.post(f"{API_URL}/search",
                      json={"query": query, "k": k, "doc_type": doc_type,
                            "date_from": date_from, "date_to": date_to}, timeout=120)
    resp.raise_for_status()
    return "\n\n".join(
        f"[{h['metadata'].get('doc_type')}] {h['metadata'].get('title') or h['metadata'].get('source')}"
        f" ({h['metadata'].get('datetime', 'n.d.')}, score {h['score']:.2f})\n{h['content'][:1200]}"
        for h in resp.json()
    ) or "no results"

In [ ]:
import datetime

from deepagents import create_deep_agent

agent = create_deep_agent(
    model,
    [search_docs],
    system_prompt=(
        # without today's date the model can't turn "recent" into a sensible date_from;
        # the date_from rule lives here because small models ignore it in tool docstrings
        f"Today is {datetime.date.today().isoformat()}. "
        "You answer questions about NHLBI BioData Catalyst (BDC). "
        "Use the search_docs tool to find relevant documentation before answering, "
        "and cite the sources you used. "
        "Search results are ranked by relevance, NOT date. For questions about recent or "
        "latest events/news you MUST pass date_from (start about a year back and widen the "
        "range if nothing returns), then pick the newest date among the results."
    ),
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is PIC-SURE and what can I do with it in BDC?"}]}
)

for m in result["messages"]:
    m.pretty_print()

In [4]:
# the tool schema exposes doc_type — see whether the model narrows the search on its own
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Find a recent BDC community hours event. When was it held?"}]}
)

for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

Find a recent BDC community hours event. When was it held?
================================== Ai Message ==================================
Tool Calls:
  search_docs (call_ruFO4nIHD8aa2rdxYAgEjfRz)
 Call ID: call_ruFO4nIHD8aa2rdxYAgEjfRz
  Args:
    query: BDC community hours event
    k: 5
    doc_type: event
================================= Tool Message =================================
Name: search_docs

[event] BDC Community Hours: What to do While Waiting for Funding (2023-04-26T13:00, score 0.58)
Join us on **[Wednesday, April 26th at 1 p.m. ET](https://bit.ly/BDC-April)** to learn how to explore BDC before you have been granted access to data and funding. You will learn how to explore BDC-hosted data prior to access and how to get started on a BDC cloud computing platform with pilot credits.

Time will be made available for discussion and to address questions, challenges, and issues you might be f

In [5]:
try:
    api.terminate()
    api.wait(timeout=10)
    print("API stopped. Server log:\n")
    print(LOG.read_text())
except NameError:
    print("no API to stop — the boot cell hasn't run in this kernel")

API stopped. Server log:

INFO:     Started server process [51368]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:58496 (Press CTRL+C to quit)
INFO:     127.0.0.1:58499 - "GET /health HTTP/1.1" 200 OK
embeddings: provider=ollama model=bge-m3 url=http://localhost:11434
INFO:     127.0.0.1:58508 - "POST /search HTTP/1.1" 200 OK
INFO:     127.0.0.1:58521 - "POST /search HTTP/1.1" 200 OK

